<a href="https://colab.research.google.com/github/erenozelll/Earthquake_Prediction/blob/main/Earthquake_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#Library Imports & Directory Initialization

import pandas as pd
import numpy as np
import os

# Ensure project directories exist for raw and processed data persistence
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print("Libraries successfully loaded and workspace directories initialized.")

Libraries successfully loaded and workspace directories initialized.


In [ ]:
# Data Ingestion and Column Standardization

file_path = 'data/raw/koeri_raw.txt'

# Load tab-separated file handling Turkish character encoding via fallback mechanism
try:
    df = pd.read_csv(file_path, sep='\t', encoding='windows-1254')
except Exception:
    df = pd.read_csv(file_path, sep='\t', encoding='iso-8859-9')

# Strip unexpected whitespaces from column headers to prevent structural errors
df.columns = df.columns.str.strip()

# Select feature subset relevant to the model and rename columns to standard snake_case
df = df[['Olus tarihi', 'Olus zamani', 'Enlem', 'Boylam', 'Der(km)', 'MD', 'ML', 'Mw', 'Yer']]
df.columns = ['Olus_tarihi', 'Olus_zamani', 'Enlem', 'Boylam', 'Der_km', 'MD', 'ML', 'Mw', 'Yer']

print("✅ Raw dataset successfully ingested with TAB delimiter.")
display(df.head())
display(df.tail())

✅ Raw dataset successfully ingested with TAB delimiter.


,Olus_tarihi,Olus_zamani,Enlem,Boylam,Der_km,MD,ML,Mw,Yer
0,2024.12.31,10:46:27.00,38.0710,37.5068,6.7,0.0,4.0,3.9,TATLAR-NURHAK (KAHRAMANMARAS) [North West 8.1...
1,2024.12.30,20:30:27.56,39.4170,37.2977,5.0,0.0,4.3,4.3,KERTMEKARACAOREN-ULAS (SIVAS) [South West 2.1...
2,2024.12.30,02:25:17.55,36.7442,30.1467,7.1,0.0,3.9,3.8,OVACIK-ELMALI (ANTALYA) [South West 6.4 km]
3,2024.12.29,00:15:10.00,35.1408,27.2722,10.6,0.0,4.1,4.3,AKDENIZ
4,2024.12.28,11:44:00.95,41.2442,44.0308,5.0,0.0,3.6,3.7,GURCISTAN


,Olus_tarihi,Olus_zamani,Enlem,Boylam,Der_km,MD,ML,Mw,Yer
12732,2000.01.03,13:54:53.10,40.90,42.06,32.0,3.6,0.0,NaN,ATLI-OLUR (ERZURUM) [East 1.8 km]
12733,2000.01.02,20:28:37.00,38.43,38.76,19.0,4.1,0.0,NaN,KALE (MALATYA) [North West 1.3 km]
12734,2000.01.02,02:16:20.00,38.83,25.51,12.0,3.5,0.0,NaN,EGE DENIZI
12735,2000.01.02,00:29:24.10,40.86,30.87,9.0,3.5,0.0,NaN,BICKIATIK-HENDEK (SAKARYA) [North East 0.9 km]
12736,2000.01.01,07:16:42.30,40.95,27.91,9.0,3.5,0.0,NaN,MARMARAEREGLISI (TEKIRDAG) [South West 4.3 km]


In [ ]:

#Data Type Casting and Hierarchical Magnitude Engineering

#Combine date and time strings into a unified, standard datetime object
df['datetime'] = pd.to_datetime(df['Olus_tarihi'] + ' ' + df['Olus_zamani'], errors='coerce')

#Enforce numeric constraints on spatial, depth, and magnitude features
numeric_cols = ['Enlem', 'Boylam', 'Der_km', 'MD', 'ML', 'Mw']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Construct a consolidated 'magnitude' feature using priority rule: Mw > ML > MD
conditions = [
    df['Mw'].notna() & (df['Mw'] > 0),
    df['ML'].notna() & (df['ML'] > 0),
    df['MD'].notna() & (df['MD'] > 0)
]
choices = [df['Mw'], df['ML'], df['MD']]
df['magnitude'] = np.select(conditions, choices, default=np.nan)

#Standardize key spatial and operational features into industry-standard English nomenclature
df = df.rename(columns={
    'Enlem': 'lat',
    'Boylam': 'lon',
    'Der_km': 'depth_km'
})

print("✅ Data type conversions and hierarchical magnitude parsing completed.")
display(df[['datetime', 'lat', 'lon', 'depth_km', 'magnitude']].head())

✅ Data type conversions and hierarchical magnitude parsing completed.


,datetime,lat,lon,depth_km,magnitude
0,2024-12-31 10:46:27.000,38.0710,37.5068,6.7,3.9
1,2024-12-30 20:30:27.560,39.4170,37.2977,5.0,4.3
2,2024-12-30 02:25:17.550,36.7442,30.1467,7.1,3.8
3,2024-12-29 00:15:10.000,35.1408,27.2722,10.6,4.3
4,2024-12-28 11:44:00.950,41.2442,44.0308,5.0,3.7


In [ ]:
# Data Quality Assurance, Spatial Filtering, and Pipeline Export

# Record the initial sample size before applying data cleaning filters
baslangic_boyutu = len(df)

# Create a isolated copy containing only the features required for machine learning
df_clean = df[['datetime', 'lat', 'lon', 'depth_km', 'magnitude']].copy()

# Drop rows containing missing (NaN) values to ensure completeness
df_clean = df_clean.dropna()

# Filter out physically impossible or extreme geological anomalies
df_clean = df_clean[
    (df_clean['magnitude'] > 0) & (df_clean['magnitude'] <= 9.0) &
    (df_clean['depth_km'] >= 0) & (df_clean['depth_km'] <= 700)
]

# Apply bounding box constraints to restrict data within the specified project coordinates (35-43°N, 25-46°E)
df_clean = df_clean[
    (df_clean['lat'] >= 35.0) & (df_clean['lat'] <= 43.0) &
    (df_clean['lon'] >= 25.0) & (df_clean['lon'] <= 46.0)
]

# Deduplicate records sharing identical timestamps and spatial coordinates
df_clean = df_clean.drop_duplicates(subset=['datetime', 'lat', 'lon'])

# Sort chronologically and reset index, which is critical for time-series validation
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

bitis_boyutu = len(df_clean)

print("--- DATA CLEANING SUMMARY ---")
print(f"Initial Row Count: {baslangic_boyutu}")
print(f"Rows Removed (Anomalies/Out of Bounds): {baslangic_boyutu - bitis_boyutu}")
print(f"Cleaned Data Ready for Training: {bitis_boyutu}")

# Export the processed dataset to CSV format for persistence
output_path = 'data/processed/koeri_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"\n Phase 1 Successful! Cleaned dataset exported to: '{output_path}'")

--- DATA CLEANING SUMMARY ---
Initial Row Count: 12737
Rows Removed (Anomalies/Out of Bounds): 16
Cleaned Data Ready for Training: 12721

 Phase 1 Successful! Cleaned dataset exported to: 'data/processed/koeri_clean.csv'
